
<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎙️ VoxCPM2 — Multilangual TTS</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Google Colab Edition — Created by <strong>The Blackbox Security</strong></h3>
  <p style='color: #ddd; margin: 0 0 0 0;'>T4 GPU (16GB VRAM) | 2B Parameters · 30 Languages · 48kHz Output · Voice Design & Cloning</p>
</div>

---

<div align="center">

  <img src="https://img.shields.io/badge/Colab-T4%20GPU-4285F4?style=for-the-badge&logo=google-colab&logoColor=white" />

  <br>

  <a href="https://www.youtube.com/@theblackboxsecurity?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>

</div>

---

### ⚠️ Quick Start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells in order
3. The Gradio UI link will appear at the bottom


In [ ]:
#@title 📦 Install Dependencies
!pip install -q voxcpm gradio soundfile numpy

In [ ]:
#@title ✅ Verify GPU & VRAM
import torch
assert torch.cuda.is_available(), "❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB")

In [ ]:
#@title 🚀 Load VoxCPM2 Model
import torch
import torch._dynamo

# Disable torch.compile globally BEFORE model loads
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.disable = True

from voxcpm import VoxCPM

print("Loading VoxCPM2 with denoiser (quality mode)...")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

model = VoxCPM.from_pretrained(
    "openbmb/VoxCPM2",
    load_denoiser=True,
    optimize=False,
)

SAMPLE_RATE = model.tts_model.sample_rate

# Warm-up
print("Warming up...")
warmup_chunks = []
for chunk in model.generate(
    text="Hello, warm up test.",
    inference_timesteps=5,
):
    warmup_chunks.append(chunk)

used = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ Model loaded! VRAM: {used:.1f} / {total:.1f} GB")
print(f"Sample rate: {SAMPLE_RATE} Hz")

In [ ]:
#@title 🎛️ Launch Gradio Interface
import gradio as gr
import soundfile as sf
import numpy as np
import tempfile
import os
import random

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">🎙️ VoxCPM2 — Multilingual TTS</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; 2B Parameters · 30 Languages · 48kHz Output · Voice Design &amp; Cloning</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""

def save_wav(wav_array: np.ndarray) -> str:
    tmp = tempfile.NamedTemporaryFile(suffix=".wav", delete=False)
    sf.write(tmp.name, wav_array, SAMPLE_RATE)
    return tmp.name

def collect_audio(generator) -> np.ndarray:
    chunks = []
    for chunk in generator:
        arr = np.asarray(chunk)
        if arr.ndim == 0:
            arr = arr.reshape(1)
        chunks.append(arr)
    if not chunks:
        raise gr.Error("Model returned no audio.")
    return np.concatenate(chunks)

def resolve_seed(seed, locked):
    if locked and seed is not None and seed >= 0:
        s = int(seed)
    else:
        s = random.randint(0, 2**31 - 1)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    return s

def tts_generate(text, cfg, steps, seed, locked):
    if not text.strip():
        raise gr.Error("Please enter some text.")
    used_seed = resolve_seed(seed, locked)
    gen = model.generate(text=text, cfg_value=cfg, inference_timesteps=int(steps))
    wav = collect_audio(gen)
    return save_wav(wav), used_seed

with gr.Blocks(theme=gr.themes.Soft(), css=CSS) as demo:
    gr.HTML(BRAND_HTML)
    gr.Markdown("## 🎤 VOICEcloner Ready")
    text = gr.Textbox(label="Enter Text")
    btn = gr.Button("Generate Voice")
    out = gr.Audio(type="filepath")

    btn.click(
        tts_generate,
        inputs=[text, gr.Number(value=2.8, visible=False), gr.Number(value=15, visible=False), gr.Number(value=-1, visible=False), gr.Checkbox(value=False, visible=False)],
        outputs=out
    )

demo.queue().launch(share=True, debug=True)


---

<div align="center">

  <a href="https://www.youtube.com/@theblackboxsecurity?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>The Blackbox Security</strong>· © All rights reserved
</p>

---
